# Scaffolded Candidate Workbench

Use this notebook after a scheduled model build has published a candidate. The analyst chooses a logical model name and friendly package number, edits the model, and supplies a business reason. Technical lineage and credentials stay inside the pipeline.


## 1. Connect

The runtime module configured for the local Cloud PC supplies SQL and Airflow settings. Running all cells is safe: execution stops at each blank required input before it can submit anything.


In [ ]:
from pricing_pipeline.workbench import Workbench

workbench = Workbench.from_runtime()
available_models = workbench.models()
display(available_models)

MODEL_NAME = ""
if not MODEL_NAME.strip():
    raise ValueError("Set MODEL_NAME explicitly to one of the model names shown above.")
if MODEL_NAME not in available_models:
    raise ValueError(f"Unknown MODEL_NAME {MODEL_NAME!r}; choose one shown above.")


## 2. Choose a candidate

The default history deliberately hides technical lineage. Only rows marked **Ready** can be opened. Enter one of their integer **Package** values explicitly.


In [ ]:
history = workbench.candidates(MODEL_NAME)
if history.empty:
    raise RuntimeError(f"No candidate history is available for {MODEL_NAME}.")

editor_ready = history.loc[history["Editor"].eq("Ready")].copy()
display(editor_ready)
if editor_ready.empty:
    raise RuntimeError(f"No editor-ready candidates are available for {MODEL_NAME}.")

PACKAGE_VERSION = None
if PACKAGE_VERSION is None:
    raise ValueError("Set PACKAGE_VERSION explicitly from the editor-ready table above.")
PACKAGE_VERSION = int(PACKAGE_VERSION)
if PACKAGE_VERSION not in set(editor_ready["Package"].astype(int)):
    raise ValueError("PACKAGE_VERSION must identify an editor-ready candidate shown above.")


## 3. Open the live editor

Keep this kernel running while editing. The live SuperGLM session in this process owns the changes that will be submitted.


In [ ]:
candidate = workbench.open(MODEL_NAME, package_version=PACKAGE_VERSION)
candidate.editor()


## 4. Submit, check status, and optionally deploy

Submitting creates an immutable child candidate in a separate Airflow run; it does not deploy anything. Replace the blank edit reason with your own market or underwriting judgement before running this cell.


In [ ]:
EDIT_REASON = ""
if not EDIT_REASON.strip():
    raise ValueError("Set EDIT_REASON explicitly before submitting editor changes.")

submission = candidate.submit_edits(
    reason=EDIT_REASON,
)


In [ ]:
status = submission.status()
status

# After status.state is 'published', enter an explicit approval reason and run manually:
# DEPLOYMENT_REASON = ""
# submission.request_deployment(reason=DEPLOYMENT_REASON)


## 5. Close the local editor

Run this cleanup cell when review is finished, before restarting the kernel, or after abandoning an edit. It stops the local widget server and discards the live session.


In [ ]:
candidate.close_editor()
